# Update meta data and objects 

In [ ]:
import scanpy as sc

import numpy as np
import pandas as pd

import os

In [ ]:
# rpy2 
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
sc.set_figure_params(figsize=(10, 10), dpi_save=1200, fontsize=18, frameon=False)

In [ ]:
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

# Set color function 
def set_color(categories): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]

    for category in categories: 

        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)

# Files and parameter 

In [ ]:
# Seurat object 
qc_h5ad_file = 'data/object/qc.h5ad'
pp_h5ad_file = 'data/object/pp.h5ad'

# Meta data 
cell_type_leiden_file = 'result/annotation/cell_type_leiden.csv'
gvhd_class_file = 'result/annotation/gvhd_class.csv'
celltypist_low_file = 'result/celltypist/immune_all_low.csv'
celltypist_high_file = 'result/celltypist/immune_all_high.csv'
genotype_file = 'result/genotype/genotype_mode_1a_genome1k.csv'
scirpy_file = 'result/scirpy/scirpy.csv'

# UMAP 
umap_file = 'data/object/components/umap.csv'

# Import 

In [ ]:
adata = sc.read_h5ad(qc_h5ad_file)

In [ ]:
cell_type_leiden = pd.read_csv(cell_type_leiden_file, index_col=0)
gvhd_class = pd.merge(adata.obs[['cell_id', 'patient_id']], pd.read_csv(gvhd_class_file), left_on='patient_id', right_on='patient_id', how='left').set_index('cell_id')[['gvhd_class']]
celltypist_low = pd.read_csv(celltypist_low_file, index_col=0).rename(columns={'predicted_labels':'celltypist_immun_low'})
celltypist_high = pd.read_csv(celltypist_high_file, index_col=0).rename(columns={'predicted_labels':'celltypist_immun_high'})
scirpy = pd.read_csv(scirpy_file, index_col=0)[['has_ir', 'cc_aa_alignment', 'cc_aa_alignment_size']]
genotype = pd.read_csv(genotype_file, index_col='cell_id')[['genotype_class']]

In [ ]:
umap = pd.read_csv(umap_file, index_col=0).to_numpy()

# Meta data 

In [ ]:
adata.obs = adata.obs.merge(cell_type_leiden, left_index=True, right_index=True, how='left')
adata.obs = adata.obs.merge(gvhd_class, left_index=True, right_index=True, how='left')
adata.obs = adata.obs.merge(celltypist_low, left_index=True, right_index=True, how='left')
adata.obs = adata.obs.merge(celltypist_high, left_index=True, right_index=True, how='left')
adata.obs = adata.obs.merge(scirpy, left_index=True, right_index=True, how='left')
adata.obs = adata.obs.merge(genotype, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['has_ir'] = adata.obs['has_ir'].astype(str).astype('category')

# UMAP

In [ ]:
adata.obsm['X_umap'] = umap

# Color 

In [ ]:
# Set colors
set_color(list(color.keys()))

# UMAP

In [ ]:
sc.pl.umap(adata, color=['cell_type_leiden', 'patient_id', 'hto_demux_class', 'time_point', 'cc_phase_class', 'digestion_protocol', 'genotype_class', 'cc_aa_alignment_size'], size=20, ncols=4, wspace=0.4)

In [ ]:
sc.pl.umap(adata, color=['celltypist_immun_high'], size=20, ncols=1, wspace=0.4)

In [ ]:
sc.pl.umap(adata, color=['celltypist_immun_low'], size=20, ncols=1, wspace=0.4)

# Save result 

In [ ]:
adata.write_h5ad(pp_h5ad_file)